# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

In dit rapport gaan wij de winkansen van voetbalwedstrijden voorspellen. Dit gaan we doen op basis van principes uit de data science en doormiddel van machine learning modellen.

---

## Importing Libraries

Als eerste moeten we altijd de juiste libraries importeren zodat we toegang hebben tot de benodigde tools.

In [1]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import os
from pathlib import Path
import tempfile

---

## Loading Data

Als volgende moeten we de data ophalen en inladen uit de `SQL` database

In [2]:
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

# Set up Kaggle credentials
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database from temp directory
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

Connected to database at: C:\Users\gamal\AppData\Local\Temp\database.sqlite


In [3]:
class Club():
    """
        Description: 
            Deze class representeert een voetbalclub. Het bevat informatie over de club en DataFrames voor spelers en wedstrijden.
        
        Attributes:
            team_id: id van de club.
            team_name: De naam van de club.
            matches_df: DataFrame met wedstrijden van de club.
            
         Methods:
            load_matches(matches_df): 
                Laadt wedstrijden van de club als DataFrame.
            get_season_matches(season): 
                Haalt wedstrijden van een bepaald seizoen.
            calculate_season_stats(season): 
                Berekent statistieken voor het team voor het gegeven seizoen.
            get_player_stats(): 
                Berekent gemiddelde en categorische statistieken voor spelers van het team.
    """
    
    def __init__(self, team_id, team_name):
        """
            Description:
                De constructor van de Club class.
                
            Args:
                team_id: id van de club.
                team_name: De naam van de club.
                
            Returns:
                None
            
        """
        self.team_id = team_id
        self.team_name = team_name
        self.matches_df = pd.DataFrame()
        
    def load_matches(self, matches_df):
        """
            Description:
                Laadt alle wedstrijden van deze club uit een matches DataFrame.
                
            Args:
                matches_df: Een DataFrame met alle wedstrijden.
                
            Returns:
                None
        """
        club_matches = matches_df[(matches_df['home_team_api_id'] == self.team_id) | (matches_df['away_team_api_id'] == self.team_id)].copy()
        self.matches_df = club_matches
        
    def get_season_matches(self, season):
        """
            Description:
                Haalt alle wedstrijden van een bepaald seizoen.
                
            Args:
                season: Het seizoen (bijv. '2015/2016').
                
            Returns:
                DataFrame met wedstrijden van dat seizoen.
        """
        return self.matches_df[self.matches_df['season'] == season]
    
    def calculate_season_stats(self, season=None):
        """
            Description:
                Berekent statistieken voor het team in een bepaald seizoen.
                
            Args:
                season: Het seizoen (optioneel). Als None, alle seizoenen.
                
            Returns:
                Dictionary met statistieken.
        """
        if season:
            matches = self.get_season_matches(season)
        else:
            matches = self.matches_df
        
        if len(matches) == 0:
            return {}
        
        wins = len(matches[
            ((matches['home_team_api_id'] == self.team_id) & (matches['home_team_goal'] > matches['away_team_goal'])) |
            ((matches['away_team_api_id'] == self.team_id) & (matches['away_team_goal'] > matches['home_team_goal']))
        ])
        
        draws = len(matches[matches['home_team_goal'] == matches['away_team_goal']])
        losses = len(matches) - wins - draws
        
        goals_for = matches[matches['home_team_api_id'] == self.team_id]['home_team_goal'].sum() + \
                    matches[matches['away_team_api_id'] == self.team_id]['away_team_goal'].sum()
        
        goals_against = matches[matches['home_team_api_id'] == self.team_id]['away_team_goal'].sum() + \
                        matches[matches['away_team_api_id'] == self.team_id]['home_team_goal'].sum()
        
        points = (wins * 3) + (draws * 1)
        
        return {
            'club': self.team_name,
            'wins': wins,
            'draws': draws,
            'losses': losses,
            'goals_for': goals_for,
            'goals_against': goals_against,
            'goal_difference': goals_for - goals_against,
            'points': points,
            'matches_played': len(matches)
        }

    def get_player_stats(self):
        """
            Description:
                Berekent gemiddelde en categorische statistieken voor spelers van het team.
                
            Returns:
                DataFrame met numerieke gemiddelden en categorische waarde tellingen.
        """
        matches = self.matches_df
            
        # Vind alle spelers die in dit seizoen hebben gespeeld in deze club
        home_players = matches[matches['home_team_api_id'] == self.team_id][['home_player_1', 'home_player_2', 'home_player_3', 'home_player_4', 'home_player_5', 'home_player_6', 'home_player_7', 'home_player_8', 'home_player_9', 'home_player_10', 'home_player_11']]
        away_players = matches[matches['away_team_api_id'] == self.team_id][['away_player_1', 'away_player_2', 'away_player_3', 'away_player_4', 'away_player_5', 'away_player_6', 'away_player_7', 'away_player_8', 'away_player_9', 'away_player_10', 'away_player_11']]     
        
        player_ids = pd.concat([home_players.stack(), away_players.stack()]).unique()
        
        global player_attributes
        
        def get_player_stats(player_id):
            player_data = player_attributes[player_attributes['player_api_id'] == player_id].drop(columns=['date'])
            return player_data.iloc[0]
        
        total_stats = []
        
        for player_id in player_ids:
            stats = get_player_stats(player_id)
            total_stats.append(stats)
        
        stats_df = pd.DataFrame(total_stats)
        
        # Calculate numeric averages
        numeric_stats = stats_df.select_dtypes(include=[np.number]).mean().round(2).to_dict()
        
        # Calculate categorical value counts and expand to separate columns
        categorical_cols = stats_df.select_dtypes(include=['object']).columns
        categorical_stats = {}
        
        for col in categorical_cols:
            value_counts = stats_df[col].value_counts().to_dict()
            for value, count in value_counts.items():
                categorical_stats[f'{col}_{value}'] = count
        
        # Combine results into single row
        result = {**numeric_stats, **categorical_stats}
        result_df = pd.DataFrame([result])
        
        return result_df.drop(columns=['id','player_api_id', 'player_fifa_api_id'], errors='ignore')
        
class Research():
    """
        Description:
            Deze class bevat methoden voor het uitvoeren van onderzoek en analyses op de club data.
        
        Methods:
            season_correlation_analysis(club, season): 
                Voert een correlatieanalyse uit op de performance van het team in een seizoen.
                
            get_season_ranking(matches_df, season, league_id):
                Geeft de ranglijst van alle teams voor een seizoen.    
    """
    
    def __init__(self):
        pass
          
    def season_correlation_analysis(self, season):
        """
            Description:
                Voert een correlatieanalyse uit op de performance van het team in een seizoen.
                Combineert team statistieken met speler statistieken en berekent correlaties met punten.
                
            Args:
                season: Het seizoen om te analyseren.
                
            Returns:
                DataFrame met correlaties gesorteerd op absolute correlatiewaarde.
        """
        global matches, teams
        
        season_matches = matches[matches['season'] == season].copy()
        
        data = []
        
        for team in season_matches['home_team_api_id'].unique():
            club = Club(team_id=team, team_name=teams.loc[teams['team_api_id'] == team, 'team_long_name'].values[0])
            club.load_matches(matches)
            
            # Get team statistics
            team_stats = club.calculate_season_stats(season)
            
            # Get player statistics
            player_stats_df = club.get_player_stats()
            player_stats = player_stats_df.iloc[0].to_dict() if not player_stats_df.empty else {}
            
            # Combine team and player stats into single row
            combined_stats = {**team_stats, **player_stats}
            data.append(combined_stats)
        
        # Convert to DataFrame
        analysis_df = pd.DataFrame(data)
        
        # Calculate correlations with points
        if 'points' not in analysis_df.columns:
            print("'points' column not found in data")
            return pd.DataFrame()
        
        correlations = []
        numeric_cols = analysis_df.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if col != 'points':  # Don't correlate points with itself
                # Skip columns with zero variance (prevents division by zero warning)
                if analysis_df[col].std() == 0:
                    continue
                
                corr_value = round(analysis_df[col].corr(analysis_df['points']), 2)
                if pd.notna(corr_value):
                    correlations.append({
                        'variable': col,
                        'correlation_with_points': corr_value
                    })
        
        # Create correlation DataFrame sorted by absolute correlation
        correlation_df = pd.DataFrame(correlations).sort_values(
            'correlation_with_points', 
            ascending=False, 
            key=abs
        ).reset_index(drop=True)
        
        # Drop irrelevant variables that are more directly calculated from points
        irrelevant_vars = ['wins', 'goal_difference', 'goals_for', 'goals_against', 'matches_played', 'draws', 'losses']
        return correlation_df[~correlation_df['variable'].isin(irrelevant_vars)]
    
    def get_season_ranking(self, matches_df, season, league_id=None):
        """
            Description:
                Geeft de ranglijst van alle teams voor een bepaald seizoen en competitie.
                
            Args:
                matches_df: DataFrame met alle wedstrijden.
                season: Het seizoen om te analyseren.
                league_id: Optional league_id.
                
            Returns:
                DataFrame met ranglijst gesorteerd op punten.
        """
        season_matches = matches_df[matches_df['season'] == season].copy()
        
        global teams
        
        if league_id:
            season_matches = season_matches[season_matches['league_id'] == league_id]
        
        if season_matches.empty:
            print(f'Geen matches gevonden voor season {season}.')
            return pd.DataFrame()
        
        season_stats_list = []
        
        for team in season_matches['home_team_api_id'].unique():
            club = Club(team_id=team, team_name=teams.loc[teams['team_api_id'] == team, 'team_long_name'].values[0])
            club.load_matches(matches_df)
            stats = club.calculate_season_stats(season)
            season_stats_list.append(stats)
        
        season_stats = pd.DataFrame(season_stats_list)
        ranking_df = season_stats.sort_values('points', ascending=False).reset_index(drop=True)
        
        return ranking_df

---

## Data Prepareren

Hier gaan we de relevante tabellen uit SQL halen en inladen in PandasDataframes om het te effectief te kunnen gebruiken.


In [4]:
teams = pd.read_sql("""
                 SELECT * 
                 FROM Team
                 """, connection)

team_attributes = pd.read_sql("""
                 SELECT * 
                 FROM Team_attributes
                 """, connection)

matches = pd.read_sql("""
                 SELECT * 
                 FROM Match
                 ---WHERE season = '2009/2010' AND league_id = 13274
                 """, connection)

matches.drop(columns=
             ['home_player_X1', 'home_player_X2', 'home_player_X3', 'home_player_X4', 'home_player_X5', 'home_player_X6', 'home_player_X7', 'home_player_X8', 'home_player_X9', 'home_player_X10', 'home_player_X11', 'away_player_X1', 'away_player_X2', 'away_player_X3', 'away_player_X4', 'away_player_X5', 'away_player_X6', 'away_player_X7', 'away_player_X8', 'away_player_X9', 'away_player_X10', 'away_player_X11', 'home_player_Y1', 'home_player_Y2', 'home_player_Y3', 'home_player_Y4', 'home_player_Y5', 'home_player_Y6', 'home_player_Y7', 'home_player_Y8', 'home_player_Y9', 'home_player_Y10', 'home_player_Y11', 'away_player_Y1', 'away_player_Y2', 'away_player_Y3', 'away_player_Y4', 'away_player_Y5', 'away_player_Y6', 'away_player_Y7', 'away_player_Y8', 'away_player_Y9', 'away_player_Y10', 'away_player_Y11', 'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 'SJH', 'SJD', 'SJA', 'VCH', 'VCD', 'VCA', 'GBH', 'GBD', 'GBA', 'BSH', 'BSD', 'BSA'], inplace=True)

league = pd.read_sql("""
                 SELECT * 
                 FROM League
                 """, connection)

league_index = {id: name for id, name in zip(league['id'], league['name'])}

players = pd.read_sql("""
                 SELECT * 
                 FROM Player
                 """, connection)

player_attributes = pd.read_sql("""
                 SELECT * 
                 FROM Player_Attributes
                 """, connection)


Elk van deze tabellen hebben kolommen die overeen komen met andere kolommen in andere tabellen. Deze kolommen worden Key Identifiers genoemd. Een key Identifier is een kolom die alleen unieke waardes heeft, zodat via andere tabellen hiernaartoe gerefereerd kan worden. Ook staan de zogenaamde linking identifiers beschreven die verschillende tabellen met elkaar linken. Dat zijn dus kolommen met waardes tussen tabellen die overeenkomen.

|  | Match | League | Player | Team | Country | Player_Attributes | Team_Attributes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| KEY IDENTIFIERS | id | id | id | id | id | id | id |
| LINKING IDENTIFIERS | country_id, league_id, home_team_api_id, away_team_api_id, home_player_1, away_player_1 | country_id | player_api_id | team_api_id |  | player_api_id | team_api_id |

---

## Data Selecteren

Voor dit onderzoek gaan we op zoek naar de statistieken van de club Sparta in het seizoen 2009/2010. Wij hebben voor het gemakkelijk en overzichtelijk inladen van functies verschillende classes en functies ontworpen.

In [5]:
# Recreate Research object with refactored classes
Onderzoek = Research()

sparta_info = teams.loc[teams['team_long_name'] == 'Sparta Rotterdam']

sparta_id = sparta_info['team_api_id'].values[0]
sparta_naam = sparta_info['team_long_name'].values[0]

# Initialize Club instance (cleaner approach - just team_id and name)
Sparta = Club(team_id=sparta_id, team_name=sparta_naam)
Sparta.load_matches(matches)

# Laat zien wat de key identifiers zijn voor onze gekozen analyse
import pandas as pd

data = {
    'Type Identifier': [
        'Club Naam (Lang)', 
        'Club Naam (Kort)', 
        'Team API ID', 
        'Team FIFA ID', 
        'League ID', 
        'Country ID', 
        'Team Attribute ID '
    ],
    'Waarde': [
        'Sparta Rotterdam', 
        'SPA', 
        10217, 
        1006, 
        13274, 
        13274, 
        1202
    ],
}

df_identifiers = pd.DataFrame(data)

display(df_identifiers)

,Type Identifier,Waarde
0,Club Naam (Lang),Sparta Rotterdam
1,Club Naam (Kort),SPA
2,Team API ID,10217
3,Team FIFA ID,1006
4,League ID,13274
5,Country ID,13274
6,Team Attribute ID,1202


---

## Interessante Statestieken

Hier doen wij onderzoek naar de statestieken van een gekozen seizoen en laten zien hoe verschillende aspecten van een team effect hebben op de uitkomst can een match en de plaatsing in een seizoen.

In [6]:
# Test season_correlation_analysis
correlations = Onderzoek.season_correlation_analysis(season='2009/2010')
print("Correlation Analysis Results for 2009/2010 Season:")
print(f"Total variables analyzed: {len(correlations)}")
print("\nTop 10 variables most correlated with points:")
display(correlations.head(10))

Correlation Analysis Results for 2009/2010 Season:
Total variables analyzed: 61

Top 10 variables most correlated with points:


,variable,correlation_with_points
4,defensive_work_rate_9,-0.61
6,dribbling,0.56
7,potential,0.56
8,short_passing,0.55
9,agility,0.53
10,long_passing,0.53
11,ball_control,0.53
12,overall_rating,0.53
13,vision,0.51
14,positioning,0.50


---

## Seizoen Ranglijst

In [7]:
display(Onderzoek.get_season_ranking(matches, season='2009/2010', league_id=13274))

,club,wins,draws,losses,goals_for,goals_against,goal_difference,points,matches_played
0,FC Twente,27,5,2,63,23,40,86,34
1,Ajax,27,4,3,106,20,86,85,34
2,PSV,23,9,2,72,29,43,78,34
3,Feyenoord,17,12,5,54,31,23,63,34
4,AZ,19,5,10,64,34,30,62,34
5,Heracles Almelo,17,5,12,54,49,5,56,34
6,FC Utrecht,14,11,9,39,33,6,53,34
7,FC Groningen,14,7,13,48,47,1,49,34
8,Roda JC Kerkrade,14,5,15,56,60,-4,47,34
9,NAC Breda,12,10,12,42,49,-7,46,34


---